<a href="https://colab.research.google.com/github/ibtihal7alharbi-tech/Computer-vision-project-/blob/main/Computer_vision_project_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:


!pip install roboflow opencv-python tensorflow -q



In [62]:

!pip install mediapipe==0.10.33 -q
!pip install urllib3 -q

In [63]:

import urllib.request
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
#tricky way to use mediapipe
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
    "pose_landmarker_full.task"
)
base_options = python.BaseOptions(model_asset_path="pose_landmarker_full.task")
print("model loaded")


print(" mediapipe:", mp.__version__)
print("IMPORT IS DONE")

model loaded
 mediapipe: 0.10.33
IMPORT IS DONE


In [64]:


from roboflow import Roboflow
rf = Roboflow(api_key="BeuKB1ot0eCALuNQFmf1")
project = rf.workspace("mohammed-o85f3").project("baby-i5wsw-a60tj")
version = project.version(1)
dataset = version.download("yolov8-obb")


loading Roboflow workspace...
loading Roboflow project...


In [65]:
import os
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# ==========================================
#No mediapipe for some uknown unsolved error
# ==========================================
IMG_SIZE = (224, 224)

def prepare_data_images(dataset_path):
    all_images = []
    all_labels = []

    images_path = os.path.join(dataset_path, "images")
    labels_path = os.path.join(dataset_path, "labels")

    for img_name in os.listdir(images_path):
        if not img_name.endswith(('.jpg', '.png', '.jpeg')):
            continue

        img_path = os.path.join(images_path, img_name)
        label_file = os.path.join(labels_path, os.path.splitext(img_name)[0] + '.txt')

        if not os.path.exists(label_file):
            continue
        with open(label_file, 'r') as f:
            line = f.readline().strip()
            if not line:
                continue
            label = int(line.split()[0])

        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, IMG_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img / 255.0  # normalize

        all_images.append(img)
        all_labels.append(label)

    print(f"  class 0 (back):    {sum(1 for l in all_labels if l == 0)}")
    print(f"  class 1 (stomach): {sum(1 for l in all_labels if l == 1)}")

    return np.array(all_images), np.array(all_labels)

print("=== Train ===")
X_train, y_train = prepare_data_images(f"{dataset.location}/train")

print("=== Validation ===")
X_val, y_val = prepare_data_images(f"{dataset.location}/valid")
# --- Load the Test Set ---
print("=== Test Set ===")
X_test, y_test = prepare_data_images(f"{dataset.location}/test")
print(f" Test: {len(X_test)}")

print(f"\n Training: {len(X_train)}")
print(f" Validation: {len(X_val)}")
print(f" Test: {len(X_test)}")

=== Train ===
  class 0 (back):    1059
  class 1 (stomach): 711
=== Validation ===
  class 0 (back):    319
  class 1 (stomach): 181
=== Test Set ===
  class 0 (back):    97
  class 1 (stomach): 167
 Test: 264

 Training: 1770
 Validation: 500
 Test: 264


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.utils import class_weight
import matplotlib.pyplot as plt

# --- 1. Handling Class Imbalance ---
# We compute weights so the model pays more attention to the 'stomach' class (which has fewer images)
weights = class_weight.compute_class_weight('balanced',
                                            classes=np.unique(y_train),
                                            y=y_train)
class_weights_dict = dict(enumerate(weights))
print(f"Computed Class Weights: {class_weights_dict}")

# --- 2. Defining the CNN Architecture ---
model = models.Sequential([
    # Layer 1: Conv + Pool (Extracts edges/lines)
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D((2, 2)),

    # Layer 2: Conv + Pool (Extracts shapes/textures)
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Layer 3: Conv + Pool (Extracts complex baby postures)
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Flattening into 1D for the Dense layers
    layers.Flatten(),

    # Fully Connected Layers
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Prevents the model from "memorizing" images
    layers.Dense(64, activation='relu'),

    # Output Layer: Softmax for 2 classes (Back vs Stomach)
    layers.Dense(2, activation='softmax')
])

# --- 3. Compiling the Model ---
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# --- 4. Training Callbacks ---
# This stops training if the model stops improving to save time
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

# --- 5. Start Training ---
print("\nStarting the training engine...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    class_weight=class_weights_dict,
    callbacks=[early_stop],
    verbose=1
)

# --- 6. Final Evaluation on Test Set ---
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n Final Test Accuracy: {test_acc*100:.2f}%")

# --- 7. Save the Model ---
model.save('baby_pose_cnn_model.h5')
print(" Model saved successfully as 'baby_pose_cnn_model.h5'")

Computed Class Weights: {0: np.float64(0.8356940509915014), 1: np.float64(1.2447257383966244)}

Starting the training engine...


Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from sklearn.utils import class_weight
import numpy as np

# --- Step 1: Data Balancing --- # We ensure that the model pays equal attention to both classes
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(weights))

# --- Step 2: Image Variation Layers (Data Augmentation) ---
# This is the secret that prevents the model from making mistakes in upside-down photos or photos taken out of bed
data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
])
# --- Step 3: Building the structure using MobileNetV2 ---
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False # Freeze the base model

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_aug,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5), # منع الحفظ (Overfitting)
    layers.Dense(2, activation='softmax')
])

# --- Step 4: Preparation and Training ---
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.00005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(" Starting Final Robust Training...")
model.fit(
    X_train, y_train,
    epochs=25,
    validation_data=(X_val, y_val),
    class_weight=class_weights_dict,
    verbose=1
)

# Save the final model
model.save('baby_guard_final_model.h5')
print(" Final Model Saved!")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# 1. Load your best model
# Ensure the filename matches what you saved
my_model = load_model('baby_pose_transfer_learning.h5')

def predict_baby_pose(image_path):
    # 2. Preprocess the image exactly like we did for training
    img = cv2.imread(image_path)
    if img is None:
        print("Invalid image path!")
        return

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224))
    img_normalized = img_resized / 255.0
    img_final = np.expand_dims(img_normalized, axis=0) # Add batch dimension

    # 3. Make the Prediction
    prediction = my_model.predict(img_final)
    class_idx = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    # 4. Map index to labels (Verify these match your yolo labels!)
    labels = {0: "Back (Safe)", 1: "Stomach (Danger!)"}
    result = labels.get(class_idx, "Unknown")

    # 5. Visualize
    plt.imshow(img_rgb)
    plt.title(f"Prediction: {result} ({confidence:.2f}%)")
    plt.axis('off')
    plt.show()

    return result, confidence

# --- TEST IT ---
# Upload an image to Colab and put the name here
# uncomment the last line then copy the path of the photo
#result, conf = predict_baby_pose('test_baby_image.jpg')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# 1. Upload the model
model_path = 'baby_guard_final_model.h5'
try:
    my_model = load_model(model_path)
    print(" Model loaded successfully!")
except:
    print(" Model file not found! Please check the path.")

def test_baby_image(image_path):

    img = cv2.imread(image_path)
    if img is None:
        print(f" Could not find image at: {image_path}")
        return
# preprocessing
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224))
    img_normalized = img_resized / 255.0  # Normalize pixels
    img_final = np.expand_dims(img_normalized, axis=0) # Add batch dimension

    # 4. (Prediction)
    prediction = my_model.predict(img_final, verbose=0)
    class_idx = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    # (Mapping)
    # 0 = Back (Safe) | 1 = Stomach (Danger)
    labels = {0: "Back (Safe) ", 1: "Stomach (Danger!) "}
    result = labels.get(class_idx, "Unknown")

    # 6. show result
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    color = 'green' if class_idx == 0 else 'red'
    plt.title(f"Result: {result}\nConfidence: {confidence:.2f}%", color=color, fontsize=14)
    plt.axis('off')
    plt.show()

    return result, confidence

#
image_path = "/content/10_JPG_jpg.rf.c88020de3d80ef6d5fd466a3f224f037.jpg"
test_baby_image(image_path)